<a href="https://colab.research.google.com/github/pakizahassan/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pakizahassan/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Logged in successfully")

Logged in successfully


In [3]:
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


In [4]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule and its reason codes

The baseline rule prioritizes content pages that show possible opportunities for refresh.

The score is based on three observed signals:

1. Low search visibility:
Pages with lower impressions may have limited search exposure and may need optimization.

2. Poor search ranking:
Pages with a high average position number are further from the top search results and may benefit from improvement.

3. Low engagement:
Pages with lower engaged sessions may indicate that users are not interacting strongly with the content.

The rule produces these reason codes:

- LOW_VISIBILITY: The page has low search impressions.
- LOW_RANKING: The page has a weak average search position.
- LOW_ENGAGEMENT: The page has lower engagement signals.
- MULTIPLE_SIGNALS: More than one issue is detected.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import pandas as pd
import numpy as np

baseline = df.copy()

# Create score
baseline["action_score"] = 0

baseline["reason_code"] = "NO_SIGNAL"


# Rule 1: Low visibility
baseline.loc[
    baseline["gsc_impressions"] < 100,
    "action_score"
] += 30


# Rule 2: Poor ranking
baseline.loc[
    baseline["gsc_avg_position"] > 20,
    "action_score"
] += 40


# Rule 3: Low engagement
baseline.loc[
    baseline["ga4_engaged_sessions"].fillna(0) < 10,
    "action_score"
] += 30


# Assign reason codes

baseline.loc[
    baseline["gsc_impressions"] < 100,
    "reason_code"
] = "LOW_VISIBILITY"


baseline.loc[
    baseline["gsc_avg_position"] > 20,
    "reason_code"
] = "LOW_RANKING"


baseline.loc[
    baseline["ga4_engaged_sessions"].fillna(0) < 10,
    "reason_code"
] = "LOW_ENGAGEMENT"


# Multiple signals
baseline.loc[
    (
        (baseline["gsc_impressions"] < 100) &
        (baseline["gsc_avg_position"] > 20)
    ),
    "reason_code"
] = "MULTIPLE_SIGNALS"


# Action label

baseline["action"] = np.where(
    baseline["action_score"] >= 50,
    "REFRESH_CONTENT",
    "MONITOR"
)


# Rank pages

baseline = baseline.sort_values(
    "action_score",
    ascending=False
)

baseline["rank"] = range(1, len(baseline)+1)


baseline.head(10)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,action_score,reason_code,action,rank
8523419,2026-03-27,client_73cda7b4e4f265ea,content_bda0372f32c70cbd,True,True,True,False,33,0,1245,...,0.0,0.0,0.0,0.0,0.0,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,1
5551982,2026-03-18,client_73cda7b4e4f265ea,content_395246f319ac2ed8,True,False,True,None,1,0,73,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,2
5551984,2026-03-18,client_73cda7b4e4f265ea,content_7faa8052b9d6529d,True,False,True,None,9,0,593,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,3
5551985,2026-03-18,client_73cda7b4e4f265ea,content_3d02a915b944c531,True,False,True,None,14,0,488,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,4
5551972,2026-03-18,client_73cda7b4e4f265ea,content_56bfee0afd84f89a,True,False,True,None,4,0,112,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,5
5551973,2026-03-18,client_73cda7b4e4f265ea,content_8b883bc57bb4527b,True,False,True,None,2,0,47,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,6
5551990,2026-03-18,client_73cda7b4e4f265ea,content_8ea7b0ac44115f68,True,False,True,None,3,0,164,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,7
2103155,2026-03-08,client_62f4a7e64f5e0096,content_03d869bbba7bbf07,True,False,True,None,11,0,400,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,8
5551993,2026-03-18,client_73cda7b4e4f265ea,content_aee62518f11ca957,True,False,True,None,25,0,1268,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,9
5551978,2026-03-18,client_73cda7b4e4f265ea,content_cf83b0f814dd6f3e,True,False,True,None,19,0,977,...,NaN,NaN,NaN,NaN,NaN,2026-03,100,MULTIPLE_SIGNALS,REFRESH_CONTENT,10


In [7]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV created successfully")

CSV created successfully


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The ranked queue prioritizes pages with multiple observed performance issues.

Confidence note:
- High confidence: Multiple signals indicate possible refresh opportunity.
- Medium confidence: One strong signal suggests improvement opportunity.
- Low confidence: Limited data is available.

Potential reasons the recommendation could be wrong:
- The page topic may be seasonal.
- Low traffic may be intentional for niche content.
- Ranking fluctuations may be temporary.
- Additional business context may change the priority.

In [8]:
top20 = baseline.head(20)

top20_review = top20[
    [
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
]

top20_review


,content_hash_id,action,reason_code,action_score,gsc_impressions,gsc_clicks,gsc_avg_position
8523419,content_bda0372f32c70cbd,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,33,0,37.727273
5551982,content_395246f319ac2ed8,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,1,0,73.000000
5551984,content_7faa8052b9d6529d,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,9,0,65.888889
5551985,content_3d02a915b944c531,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,14,0,34.857143
5551972,content_56bfee0afd84f89a,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,4,0,28.000000
5551973,content_8b883bc57bb4527b,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,2,0,23.500000
5551990,content_8ea7b0ac44115f68,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,3,0,54.666667
2103155,content_03d869bbba7bbf07,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,11,0,36.363636
5551993,content_aee62518f11ca957,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,25,0,50.720000
5551978,content_cf83b0f814dd6f3e,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,19,0,51.421053


In [10]:
top20_review = top20[
    [
        "content_hash_id",
        "action",
        "reason_code",
        "action_score"
    ]
].copy()


top20_review["confidence_note"] = (
    "High confidence because multiple observed signals indicate opportunity"
)


top20_review["what_would_make_it_wrong"] = (
    "Seasonality, niche audience, temporary ranking changes, or missing context"
)


top20_review

,content_hash_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
8523419,content_bda0372f32c70cbd,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551982,content_395246f319ac2ed8,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551984,content_7faa8052b9d6529d,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551985,content_3d02a915b944c531,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551972,content_56bfee0afd84f89a,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551973,content_8b883bc57bb4527b,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551990,content_8ea7b0ac44115f68,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
2103155,content_03d869bbba7bbf07,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551993,content_aee62518f11ca957,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."
5551978,content_cf83b0f814dd6f3e,REFRESH_CONTENT,MULTIPLE_SIGNALS,100,High confidence because multiple observed sign...,"Seasonality, niche audience, temporary ranking..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may be weak because the baseline is a simple rule-based approach.

Examples of weak picks:
- A page with low impressions may intentionally target a small audience.
- A page with poor ranking may need technical SEO improvements instead of content refresh.
- A page with low clicks may require a better title or metadata rather than rewriting content.

Leakage check:

The baseline uses only historical information available before making a decision:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_engaged_sessions

No future performance metrics, labels, or outcome-derived features were used.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
import os

os.path.exists("work/outputs/baseline_action_score.csv")

True

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.